# Regression NBA Model


## Configuration

## Imports

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from nba_ou.data_preparation.missing_data.clean_df_for_training import (
    clean_dataframe_for_training,
)
from nba_ou.modeling.modeling import (
    TemporalDecaySampleWeightRegressor,
    evaluate_day_by_day_walk_forward,
    split_latest_dates_holdout,
    make_walk_forward_last_n_seasons_splits,
    validate_time_splits,
    make_test_anchored_walk_forward_splits,
    assert_valid_time_splits,
    save_model_bundle,
    load_model_bundle,
)


In [2]:
TARGET_COL = "LINE_ERROR"
SAMPLE_WEIGHT_LAMBDA = 0.005
SAMPLE_WEIGHT_LAMBDA_BOUNDS = (1e-4, 0.01)
TRAIN_GAMES = 6750

## Load Data

In [3]:
nan_threshold = 5
max_na_per_row = 3


data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "all_odds_training_data_until_20260405.csv"

path = data_path + name

header_cols = pd.read_csv(path, nrows=0).columns
dtype_dict = {col: str for col in header_cols if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict,
)
df_stats["GAME_DATE"] = pd.to_datetime(df_stats["GAME_DATE"]).dt.strftime("%Y-%m-%d")


In [4]:
exclude = "fanatics_sportsbook"

In [5]:
df_to_train = clean_dataframe_for_training(df_stats, nan_threshold=nan_threshold, max_na_per_row=max_na_per_row, create_missing_flags=False, verbose=1, keep_columns=['GAME_DATE'], exclude_cols_containing=[exclude])

STARTING DATAFRAME CLEANING PIPELINE
Starting basic cleaning with 11383 rows
Basic cleaning complete: 8774 rows remaining

Starting advanced column cleaning with 2948 columns

Advanced column cleaning complete: 2948 → 1369 columns (1579 removed)


Applying missing data policy...

Missing Data Policy Report:
  Rows dropped: 0 (0.0%)
  Critical columns requiring data: 4
  Columns zero-filled: 104
  Infer pairs applied: 20/136
  Remaining NaN cells: 133425

Dropping rows with more than 3 NaN values...
Removed 1508 rows exceeding NaN threshold
CLEANING COMPLETE
Final shape: (7266, 1369)


In [6]:
import time
time.sleep(5.7*3600)

In [7]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        null_rows = df_to_train[df_to_train[col].isna()]
        if len(null_rows) > 0 and "SEASON_YEAR" in df_to_train.columns:
            common_season = null_rows["SEASON_YEAR"].mode()
            most_common_season.append(
                common_season.iloc[0] if len(common_season) > 0 else None
            )
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame(
    {
        "Column": na_counts.index,
        "NA_Count": na_counts.values,
        "NA_Percentage": (na_counts.values / len(df_to_train) * 100).round(2),
        "Most_Common_Season_Year": most_common_season,
    }
).sort_values("NA_Count", ascending=False)

na_counts_df[na_counts_df["NA_Count"] > 0]

,Column,NA_Count,NA_Percentage,Most_Common_Season_Year
1167,LEAGUE_GAMES_LAST_1D_BEFORE,266,3.66,2023.0
1359,TRAVEL_RECENCY_RATIO_AWAY_2D_OVER_14D_BEFORE,79,1.09,2019.0
210,ml_betmgm_price_LAST_ALL_5_MATCHES_BEFORE_TEAM...,62,0.85,2019.0
629,ml_betmgm_price_LAST_ALL_5_MATCHES_BEFORE_TEAM...,60,0.83,2019.0
1358,TRAVEL_RECENCY_RATIO_HOME_2D_OVER_14D_BEFORE,41,0.56,2019.0
895,DIFF_FROM_LINE_caesars_LAST_ALL_1_MATCHES_DIFF...,28,0.39,2023.0
538,DIFF_FROM_LINE_caesars_LAST_ALL_1_MATCHES_BEFO...,22,0.30,2023.0
121,DIFF_FROM_LINE_caesars_LAST_ALL_1_MATCHES_BEFO...,15,0.21,2023.0
902,DIFF_FROM_LINE_draftkings_LAST_ALL_1_MATCHES_D...,8,0.11,2020.0
1207,ml_consensus_opener_price_away,8,0.11,2025.0


In [8]:
BET365_LINE_COL = "TOTAL_LINE_bet365"
# BET365_LINE_COL = "total_bet365_line_over"

# Ensure the main scoring line and actual total exist.
df_to_train = df_to_train.dropna(subset=[BET365_LINE_COL, "TOTAL_POINTS"]).copy()

In [9]:
df_to_train["LINE_ERROR"] = df_to_train["TOTAL_POINTS"] - df_to_train[BET365_LINE_COL]


In [10]:
df_to_train["GAME_DATE"] = pd.to_datetime(df_to_train["GAME_DATE"])
df_to_train = df_to_train.sort_values("GAME_DATE").reset_index(drop=True)

# Count games per season
games_per_season = df_to_train.groupby("SEASON_YEAR").size()
print(games_per_season)


SEASON_YEAR
2019     296
2020    1071
2021    1221
2022    1202
2023    1169
2024    1232
2025    1075
dtype: int64


## Train / Test

In [11]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
)
from sklearn.model_selection import cross_validate
from xgboost import XGBRegressor

from nba_ou.modeling.optuna_error_line import (
    fit_best_xgb_error_line,
    select_best_trial_lexicographic,
    summarize_lexicographic_candidates,
    summarize_optuna_trials,
    tune_xgb_error_line_optuna,
)
from nba_ou.modeling.scorers import (
    OverUnderScorerLineError,
    OverUnderScorerLineErrorMinEdge,
    evaluate_error_thresholds,
    over_under_betting_accuracy_error_line,
    over_under_betting_accuracy_error_line_with_min_edge,
)

In [12]:
df_dev, df_test_final = split_latest_dates_holdout(
    df=df_to_train,
    date_col="GAME_DATE",
    test_size=0.04,
)

print(f"Development set size: {len(df_dev)}")
print(f"Final test set size: {len(df_test_final)}")
print(
    "Final test date range:",
    df_test_final["GAME_DATE"].min(),
    "->",
    df_test_final["GAME_DATE"].max(),
)

Development set size: 6967
Final test set size: 299
Final test date range: 2026-02-24 00:00:00 -> 2026-04-04 00:00:00


In [13]:
def build_recency_sample_weights(df, date_col="GAME_DATE", lambda_=SAMPLE_WEIGHT_LAMBDA):
    dates = pd.to_datetime(df[date_col])
    max_date = dates.max()
    age_days = (max_date - dates).dt.days
    weights = np.exp(-lambda_ * age_days)
    return pd.Series(weights, index=df.index, name="sample_weight")

EXCLUDE_COLS = [
    "TOTAL_POINTS",
    "LINE_ERROR",
    "SEASON_YEAR",
    "GAME_DATE",
]

X_dev = df_dev.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(df_dev[TARGET_COL], errors="coerce")
sample_weight_dev = build_recency_sample_weights(df_dev)

X_test_final = df_test_final.drop(columns=EXCLUDE_COLS, errors="ignore")
y_test_final = pd.to_numeric(df_test_final[TARGET_COL], errors="coerce")

print(f"X_dev shape: {X_dev.shape}")
print(f"X_test_final shape: {X_test_final.shape}")
print(
    f"Recency sample weights lambda={SAMPLE_WEIGHT_LAMBDA}: "
    f"min={sample_weight_dev.min():.4f}, max={sample_weight_dev.max():.4f}"
)


X_dev shape: (6967, 1366)
X_test_final shape: (299, 1366)
Recency sample weights lambda=0.005: min=0.0000, max=1.0000


In [14]:
ou_scorer = OverUnderScorerLineError()
ou_scorer_edge_2 = OverUnderScorerLineErrorMinEdge(min_edge=2)
ou_scorer_edge_4 = OverUnderScorerLineErrorMinEdge(min_edge=4)

scoring = {
    "MSE": "neg_mean_squared_error",
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2",
    "OU_Betting_Accuracy": ou_scorer,
    "OU_Betting_Accuracy_Edge_2": ou_scorer_edge_2,
    "OU_Betting_Accuracy_Edge_4": ou_scorer_edge_4,
}


def print_metrics(cv_results):
    for sc in scoring.keys():
        train_key = f"train_{sc}"
        test_key = f"test_{sc}"

        train_val = cv_results[train_key].mean()
        test_val = cv_results[test_key].mean()

        if sc in {"MSE", "RMSE", "MAE"}:
            train_val = -train_val
            test_val = -test_val

        if sc.startswith("OU_Betting_Accuracy"):
            print(f"Train {sc}: {train_val:.2%}")
            print(f"Validation {sc}: {test_val:.2%}")
        else:
            print(f"Train {sc}: {train_val:.5f}")
            print(f"Validation {sc}: {test_val:.5f}")
        print()


In [15]:
DAY_BY_DAY_METRIC_NAME = "OU_Betting_Accuracy"
DAY_BY_DAY_THRESHOLDS = (1, 2, 3)


def summarize_walk_forward_thresholds(predictions_df, thresholds=DAY_BY_DAY_THRESHOLDS):
    y_true = pd.to_numeric(predictions_df["y_true"], errors="coerce").to_numpy(dtype=float)
    y_pred = pd.to_numeric(predictions_df["y_pred"], errors="coerce").to_numpy(dtype=float)
    margin = np.abs(y_pred)
    n_total = len(predictions_df)

    rows = []
    for t in thresholds:
        mask = margin > t
        n = int(mask.sum())
        acc = (
            np.nan
            if n == 0
            else over_under_betting_accuracy_error_line(
                y_true_error=y_true[mask],
                y_pred_error=y_pred[mask],
            )
        )
        rows.append(
            {
                "threshold_abs_pred_error_gt": t,
                "n_games": n,
                "pct_of_test": (n / n_total) if n_total else np.nan,
                "directional_accuracy": acc,
            }
        )

    return pd.DataFrame(rows)


def run_day_by_day_walk_forward_evaluation(
    *,
    label,
    df_dev,
    df_test_final,
    fit_and_predict,
    max_games=TRAIN_GAMES,
    metric_name=DAY_BY_DAY_METRIC_NAME,
    thresholds=DAY_BY_DAY_THRESHOLDS,
):
    result = evaluate_day_by_day_walk_forward(
        df_dev=df_dev,
        df_test_final=df_test_final,
        fit_and_predict=fit_and_predict,
        metric_fn=lambda y_true, y_pred: over_under_betting_accuracy_error_line(
            y_true_error=y_true,
            y_pred_error=y_pred,
        ),
        target_col=TARGET_COL,
        max_games=max_games,
        metric_name=metric_name,
    )

    threshold_results = summarize_walk_forward_thresholds(
        result.predictions,
        thresholds=thresholds,
    )

    print(f"{label} mean day-by-day {metric_name}: {result.mean_metric:.2%}")
    display(result.daily_results.style.format({metric_name: "{:.2%}"}))
    print(f"{label} thresholded walk-forward accuracy")
    display(
        threshold_results.style.format(
            {"pct_of_test": "{:.1%}", "directional_accuracy": "{:.2%}"}
        )
    )
    return result, threshold_results


In [16]:
splits, fold_info = make_test_anchored_walk_forward_splits(
    df=df_dev,
    date_col="GAME_DATE",
    season_col="SEASON_YEAR",
    test_games=25,
    step_games_between_tests=25,
    train_games=TRAIN_GAMES,
    min_train_games=TRAIN_GAMES*0.8,
    max_folds=15,
    verbose=1,
)

assert_valid_time_splits(df_dev, splits)



Created 15 test-anchored walk-forward folds
 fold  train_n_games  test_n_games train_start_date train_end_date test_start_date test_end_date  test_season
    1           6044            27       2019-11-10     2025-04-04      2025-04-05    2025-04-08         2024
    2           6100            25       2019-11-10     2025-04-11      2025-04-13    2025-04-23         2024
    3           6191            32       2019-11-10     2025-06-22      2025-10-27    2025-11-03         2025
    4           6252            30       2019-11-10     2025-11-07      2025-11-08    2025-11-11         2025
    5           6311            31       2019-11-10     2025-11-15      2025-11-16    2025-11-19         2025
    6           6370            33       2019-11-10     2025-11-23      2025-11-24    2025-11-28         2025
    7           6428            32       2019-11-10     2025-12-01      2025-12-02    2025-12-05         2025
    8           6485            36       2019-11-10     2025-12-11      2025

In [17]:
season_bl = DummyRegressor(strategy="mean")

cv_results = cross_validate(
    season_bl,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("DummyRegressor baseline")
print_metrics(cv_results)


DummyRegressor baseline
Train MSE: 293.85362
Validation MSE: 265.64048

Train RMSE: 17.14215
Validation RMSE: 16.15563

Train MAE: 13.65495
Validation MAE: 13.13850

Train R2: 0.00000
Validation R2: -0.02943

Train OU_Betting_Accuracy: 51.74%
Validation OU_Betting_Accuracy: 50.65%

Train OU_Betting_Accuracy_Edge_2: 0.00%
Validation OU_Betting_Accuracy_Edge_2: 0.00%

Train OU_Betting_Accuracy_Edge_4: 0.00%
Validation OU_Betting_Accuracy_Edge_4: 0.00%



In [18]:
lr = LinearRegression()

cv_results = cross_validate(
    lr,
    X_dev.fillna(0),
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1,
)

print("Linear Regression")
print_metrics(cv_results)


Linear Regression
Train MSE: 235.76354
Validation MSE: 3845.46066

Train RMSE: 15.35430
Validation RMSE: 36.78092

Train MAE: 12.16332
Validation MAE: 24.24413

Train R2: 0.19768
Validation R2: -11.99467

Train OU_Betting_Accuracy: 64.41%
Validation OU_Betting_Accuracy: 51.34%

Train OU_Betting_Accuracy_Edge_2: 68.27%
Validation OU_Betting_Accuracy_Edge_2: 51.73%

Train OU_Betting_Accuracy_Edge_4: 72.11%
Validation OU_Betting_Accuracy_Edge_4: 52.75%



In [19]:
xgb_reg_no_weights = XGBRegressor(
    max_depth=3,
    learning_rate=0.05,
    n_estimators=100,
    subsample=0.65,
    colsample_bytree=0.68,
    reg_alpha=5.28,
    reg_lambda=1.3,
    min_child_weight=5.08,
    gamma=0.0085,
    n_jobs=-1,
    random_state=16,
)

cv_results_no_weights = cross_validate(
    xgb_reg_no_weights,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("XGBoost no sample weights")
print_metrics(cv_results_no_weights)

XGBoost no sample weights
Train MSE: 255.51622
Validation MSE: 271.64168

Train RMSE: 15.98483
Validation RMSE: 16.35208

Train MAE: 12.72002
Validation MAE: 13.28427

Train R2: 0.13046
Validation R2: -0.05857

Train OU_Betting_Accuracy: 67.96%
Validation OU_Betting_Accuracy: 51.79%

Train OU_Betting_Accuracy_Edge_2: 83.73%
Validation OU_Betting_Accuracy_Edge_2: 47.77%

Train OU_Betting_Accuracy_Edge_4: 93.91%
Validation OU_Betting_Accuracy_Edge_4: 25.76%



In [20]:
xgb_reg_no_weights.fit(X_dev, y_dev)

y_pred_test_error = xgb_reg_no_weights.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_error)
rmse = root_mean_squared_error(y_test_final, y_pred_test_error)
mae = mean_absolute_error(y_test_final, y_pred_test_error)
ou_acc = over_under_betting_accuracy_error_line(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
)
ou_acc_edge_2 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=2,
)
ou_acc_edge_4 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=4,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")
print(f"OU_Betting_Accuracy_Edge_2: {ou_acc_edge_2:.2%}")
print(f"OU_Betting_Accuracy_Edge_4: {ou_acc_edge_4:.2%}")

Final test metrics
MSE: 311.21908
RMSE: 17.64140
MAE: 13.70445
OU_Betting_Accuracy: 52.56%
OU_Betting_Accuracy_Edge_2: 57.14%
OU_Betting_Accuracy_Edge_4: 55.56%


In [21]:
results_df, y_pred_test_error = evaluate_error_thresholds(
    model=xgb_reg_no_weights,
    X_test=X_test_final,
    y_test_error=y_test_final,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "directional_accuracy": "{:.2%}"}
    )
)


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,0,299,100.0%,52.56%
1,1,182,60.9%,55.56%
2,2,99,33.1%,57.14%
3,3,34,11.4%,48.48%
4,4,9,3.0%,55.56%
5,5,1,0.3%,100.00%
6,6,0,0.0%,nan%
7,7,0,0.0%,nan%
8,8,0,0.0%,nan%
9,9,0,0.0%,nan%


In [22]:
def fit_and_predict_xgb_no_weights_day_by_day(train_df, test_df):
    model = XGBRegressor(**xgb_reg_no_weights.get_params())

    X_train = train_df.drop(columns=EXCLUDE_COLS, errors="ignore")
    y_train = pd.to_numeric(train_df[TARGET_COL], errors="coerce")
    X_test = test_df.drop(columns=EXCLUDE_COLS, errors="ignore")

    model.fit(X_train, y_train)
    return model.predict(X_test)


day_by_day_no_weights, day_by_day_no_weights_thresholds = run_day_by_day_walk_forward_evaluation(
    label="XGBoost no sample weights",
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_xgb_no_weights_day_by_day,
    max_games=TRAIN_GAMES,
)

XGBoost no sample weights mean day-by-day OU_Betting_Accuracy: 51.28%


,date,train_n_games,test_n_games,train_start_date,train_end_date,OU_Betting_Accuracy
0,2026-02-24 00:00:00,6750,11,2020-03-08 00:00:00,2026-02-23 00:00:00,27.27%
1,2026-02-25 00:00:00,6750,6,2020-03-10 00:00:00,2026-02-24 00:00:00,33.33%
2,2026-02-26 00:00:00,6750,10,2020-03-10 00:00:00,2026-02-25 00:00:00,33.33%
3,2026-02-27 00:00:00,6750,5,2020-07-31 00:00:00,2026-02-26 00:00:00,40.00%
4,2026-02-28 00:00:00,6750,5,2020-08-01 00:00:00,2026-02-27 00:00:00,60.00%
5,2026-03-01 00:00:00,6750,11,2020-08-02 00:00:00,2026-02-28 00:00:00,54.55%
6,2026-03-02 00:00:00,6750,4,2020-08-04 00:00:00,2026-03-01 00:00:00,100.00%
7,2026-03-03 00:00:00,6750,10,2020-08-05 00:00:00,2026-03-02 00:00:00,60.00%
8,2026-03-04 00:00:00,6750,5,2020-08-06 00:00:00,2026-03-03 00:00:00,40.00%
9,2026-03-05 00:00:00,6750,9,2020-08-07 00:00:00,2026-03-04 00:00:00,66.67%


XGBoost no sample weights thresholded walk-forward accuracy


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,1,178,59.5%,50.00%
1,2,83,27.8%,60.24%
2,3,34,11.4%,50.00%


## Check weighted

In [23]:
xgb_reg_weights = XGBRegressor(
    max_depth=3,
    learning_rate=0.05,
    n_estimators=35,
    subsample=0.65,
    colsample_bytree=0.68,
    reg_alpha=5.28,
    reg_lambda=1.3,
    min_child_weight=5.08,
    gamma=0.0085,
    n_jobs=-1,
    random_state=16,
)

weighted_xgb = TemporalDecaySampleWeightRegressor(
    estimator=xgb_reg_weights,
    dates=df_dev["GAME_DATE"],
    lambda_=SAMPLE_WEIGHT_LAMBDA,
)

cv_results_weights = cross_validate(
    weighted_xgb,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("XGBoost with sample weights (per-fold decay)")
print_metrics(cv_results_weights)

XGBoost with sample weights (per-fold decay)
Train MSE: 285.39431
Validation MSE: 269.37361

Train RMSE: 16.89358
Validation RMSE: 16.26538

Train MAE: 13.43259
Validation MAE: 13.25061

Train R2: 0.02878
Validation R2: -0.04604

Train OU_Betting_Accuracy: 56.08%
Validation OU_Betting_Accuracy: 51.77%

Train OU_Betting_Accuracy_Edge_2: 63.85%
Validation OU_Betting_Accuracy_Edge_2: 48.01%

Train OU_Betting_Accuracy_Edge_4: 80.53%
Validation OU_Betting_Accuracy_Edge_4: 40.68%



In [24]:
weighted_xgb.fit(X_dev, y_dev)

y_pred_test_error = weighted_xgb.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_error)
rmse = root_mean_squared_error(y_test_final, y_pred_test_error)
mae = mean_absolute_error(y_test_final, y_pred_test_error)
ou_acc = over_under_betting_accuracy_error_line(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
)
ou_acc_edge_2 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=2,
)
ou_acc_edge_4 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=4,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")
print(f"OU_Betting_Accuracy_Edge_2: {ou_acc_edge_2:.2%}")
print(f"OU_Betting_Accuracy_Edge_4: {ou_acc_edge_4:.2%}")

Final test metrics
MSE: 309.44161
RMSE: 17.59095
MAE: 13.63003
OU_Betting_Accuracy: 51.19%
OU_Betting_Accuracy_Edge_2: 58.25%
OU_Betting_Accuracy_Edge_4: 68.42%


In [25]:
results_df, y_pred_test_error = evaluate_error_thresholds(
    model=weighted_xgb,
    X_test=X_test_final,
    y_test_error=y_test_final,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "directional_accuracy": "{:.2%}"}
    )
)


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,0,299,100.0%,51.19%
1,1,194,64.9%,55.50%
2,2,104,34.8%,58.25%
3,3,57,19.1%,61.40%
4,4,19,6.4%,68.42%
5,5,6,2.0%,66.67%
6,6,2,0.7%,50.00%
7,7,0,0.0%,nan%
8,8,0,0.0%,nan%
9,9,0,0.0%,nan%


In [26]:
def fit_and_predict_xgb_weights_day_by_day(train_df, test_df):
    base_model = XGBRegressor(**xgb_reg_weights.get_params())
    model = TemporalDecaySampleWeightRegressor(
        estimator=base_model,
        dates=train_df["GAME_DATE"],
        lambda_=SAMPLE_WEIGHT_LAMBDA,
    )

    X_train = train_df.drop(columns=EXCLUDE_COLS, errors="ignore")
    y_train = pd.to_numeric(train_df[TARGET_COL], errors="coerce")
    X_test = test_df.drop(columns=EXCLUDE_COLS, errors="ignore")

    model.fit(X_train, y_train)
    return model.predict(X_test)


day_by_day_weights, day_by_day_weights_thresholds = run_day_by_day_walk_forward_evaluation(
    label="XGBoost with sample weights",
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_xgb_weights_day_by_day,
    max_games=TRAIN_GAMES,
)


XGBoost with sample weights mean day-by-day OU_Betting_Accuracy: 52.27%


,date,train_n_games,test_n_games,train_start_date,train_end_date,OU_Betting_Accuracy
0,2026-02-24 00:00:00,6750,11,2020-03-08 00:00:00,2026-02-23 00:00:00,90.91%
1,2026-02-25 00:00:00,6750,6,2020-03-10 00:00:00,2026-02-24 00:00:00,16.67%
2,2026-02-26 00:00:00,6750,10,2020-03-10 00:00:00,2026-02-25 00:00:00,44.44%
3,2026-02-27 00:00:00,6750,5,2020-07-31 00:00:00,2026-02-26 00:00:00,40.00%
4,2026-02-28 00:00:00,6750,5,2020-08-01 00:00:00,2026-02-27 00:00:00,60.00%
5,2026-03-01 00:00:00,6750,11,2020-08-02 00:00:00,2026-02-28 00:00:00,63.64%
6,2026-03-02 00:00:00,6750,4,2020-08-04 00:00:00,2026-03-01 00:00:00,50.00%
7,2026-03-03 00:00:00,6750,10,2020-08-05 00:00:00,2026-03-02 00:00:00,80.00%
8,2026-03-04 00:00:00,6750,5,2020-08-06 00:00:00,2026-03-03 00:00:00,40.00%
9,2026-03-05 00:00:00,6750,9,2020-08-07 00:00:00,2026-03-04 00:00:00,77.78%


XGBoost with sample weights thresholded walk-forward accuracy


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,1,203,67.9%,55.33%
1,2,130,43.5%,53.12%
2,3,71,23.7%,59.42%


# Optuna

In [ ]:
study = tune_xgb_error_line_optuna(
    X=X_dev,
    y=y_dev,
    sample_weight_dates=df_dev["GAME_DATE"],
    tune_sample_weight_lambda=True,
    sample_weight_lambda_bounds=SAMPLE_WEIGHT_LAMBDA_BOUNDS,
    splits=splits,
    n_trials=80,
    timeout=4.5 * 3600,
    # timeout=600,

    objective_name="reg:squarederror",
    study_name="xgb_error_line_mae",
)

best_trial_lexi = select_best_trial_lexicographic(
    study,
    mae_tolerance_abs=0.15,
)

print("Optuna best by MAE only")
print("Trial:", study.best_trial.number)
print("Best CV MAE:", study.best_value)
print("Mean OU accuracy:", study.best_trial.user_attrs.get("mean_ou_acc"))
print("Mean OU accuracy edge 2:", study.best_trial.user_attrs.get("mean_ou_acc_edge_2"))
print("Mean OU accuracy edge 4:", study.best_trial.user_attrs.get("mean_ou_acc_edge_4"))

print("\nSelected trial after MAE-first / OU-second ranking")
print("Trial:", best_trial_lexi.number)
print("CV MAE:", best_trial_lexi.user_attrs.get("mean_mae", best_trial_lexi.value))
print("Mean RMSE:", best_trial_lexi.user_attrs.get("mean_rmse"))
print("Mean R2:", best_trial_lexi.user_attrs.get("mean_r2"))
print("Mean OU accuracy:", best_trial_lexi.user_attrs.get("mean_ou_acc"))
print("Mean OU accuracy edge 2:", best_trial_lexi.user_attrs.get("mean_ou_acc_edge_2"))
print("Mean OU accuracy edge 4:", best_trial_lexi.user_attrs.get("mean_ou_acc_edge_4"))
print("Median best_iteration:", best_trial_lexi.user_attrs.get("median_best_iteration"))
print("Params:")
for k, v in best_trial_lexi.params.items():
    print(f"{k}: {v}")

trials_df = summarize_optuna_trials(study)
display(
    trials_df.head(15).style.format(
        {
            "value_mae": "{:.4f}",
            "mean_rmse": "{:.4f}",
            "mean_r2": "{:.4f}",
            "mean_ou_acc": "{:.2%}",
            "mean_ou_acc_edge_2": "{:.2%}",
            "mean_ou_acc_edge_4": "{:.2%}",
        }
    )
)

candidates_df = summarize_lexicographic_candidates(
    study,
    mae_tolerance_abs=0.031,
)
display(
    candidates_df.head(15).style.format(
        {
            "value_mae": "{:.4f}",
            "mean_mae": "{:.4f}",
            "mean_rmse": "{:.4f}",
            "mean_r2": "{:.4f}",
            "mean_ou_acc": "{:.2%}",
            "mean_ou_acc_edge_2": "{:.2%}",
            "mean_ou_acc_edge_4": "{:.2%}",
        }
    )
)


[I 2026-04-06 04:46:03,058] A new study created in memory with name: xgb_error_line_mae


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-04-06 04:52:15,385] Trial 0 finished with value: 13.03064380426177 and parameters: {'max_depth': 2, 'min_child_weight': 18.346704707583235, 'gamma': 1.6970342240854253, 'subsample': 0.5682407800531226, 'colsample_bytree': 0.5123279759064977, 'learning_rate': 0.011926786034588454, 'reg_alpha': 1.8771791376898666, 'reg_lambda': 1.897469395521307, 'sample_weight_lambda': 0.00013824509574177668}. Best is trial 0 with value: 13.03064380426177.
[I 2026-04-06 04:59:46,503] Trial 1 finished with value: 13.014894284354575 and parameters: {'max_depth': 4, 'min_child_weight': 20.290108931287893, 'gamma': 0.3261777842309625, 'subsample': 0.8390562044499359, 'colsample_bytree': 0.4213034780854249, 'learning_rate': 0.012620826760486504, 'reg_alpha': 0.09307011182812809, 'reg_lambda': 15.258811505244246, 'sample_weight_lambda': 0.000848258413826022}. Best is trial 1 with value: 13.014894284354575.
[I 2026-04-06 05:04:58,066] Trial 2 finished with value: 12.940971728149416 and parameters: {'ma

,trial,value_mae,mean_rmse,mean_r2,mean_ou_acc,mean_ou_acc_edge_2,mean_ou_acc_edge_3,mean_ou_acc_edge_4,mean_best_iteration,median_best_iteration,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda,sample_weight_lambda
0,22,12.7419,15.8464,0.0082,56.36%,44.78%,0.391138,27.61%,43,19,2,6.815241,2.003961,0.634872,0.755911,0.059293,0.667165,8.055502,0.006799
1,42,12.7760,15.8445,0.0070,56.78%,53.18%,0.541725,30.50%,61,27,2,6.193310,2.456921,0.610167,0.732671,0.052739,0.357460,6.044555,0.004725
2,16,12.7772,15.8471,0.0071,54.84%,49.38%,0.427751,36.21%,54,25,3,5.708433,2.454432,0.641687,0.736418,0.036388,0.949922,5.379128,0.007130
3,31,12.7787,15.8321,0.0092,56.06%,54.05%,0.368822,38.78%,77,58,2,10.154922,2.539232,0.627916,0.748093,0.037696,0.549346,6.282468,0.004852
4,12,12.7788,15.8016,0.0127,58.79%,42.74%,0.450278,38.50%,72,43,3,8.380570,2.266313,0.680355,0.798793,0.052860,17.032557,3.676522,0.002316
5,48,12.7818,15.8594,0.0068,54.39%,50.85%,0.410839,26.59%,52,11,2,9.451883,2.408085,0.692916,0.681982,0.042157,0.081181,4.110317,0.008937
6,32,12.8004,15.8330,0.0103,54.71%,48.06%,0.495149,56.73%,76,59,2,6.146779,2.362444,0.583205,0.768137,0.051387,0.355657,11.523724,0.004563
7,17,12.8023,15.8455,0.0083,55.00%,45.56%,0.338971,38.46%,74,29,2,10.286326,2.561929,0.627907,0.733404,0.038891,0.540867,6.737132,0.005956
8,47,12.8048,15.8848,0.0051,56.22%,61.85%,0.411760,31.81%,34,10,3,24.571209,2.150971,0.572317,0.721357,0.054521,0.659975,2.793758,0.003758
9,23,12.8073,15.9690,-0.0078,56.57%,52.93%,0.405205,24.22%,27,9,2,6.756782,2.046344,0.597486,0.765385,0.058389,0.656987,8.763033,0.007426


,trial,value_mae,mean_mae,mean_rmse,mean_r2,mean_ou_acc,mean_ou_acc_edge_2,mean_ou_acc_edge_3,mean_ou_acc_edge_4,mean_best_iteration,median_best_iteration,mae_cutoff,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda,sample_weight_lambda
0,12,12.7788,12.7788,15.8016,0.0127,58.79%,42.74%,0.450278,38.50%,72,43,12.891907,3,8.380570,2.266313,0.680355,0.798793,0.052860,17.032557,3.676522,0.002316
1,14,12.8508,12.8508,15.9491,-0.0044,57.07%,51.36%,0.351410,25.56%,51,17,12.891907,3,5.100975,2.198923,0.733962,0.548776,0.027712,6.741589,4.102285,0.009918
2,42,12.7760,12.7760,15.8445,0.0070,56.78%,53.18%,0.541725,30.50%,61,27,12.891907,2,6.193310,2.456921,0.610167,0.732671,0.052739,0.357460,6.044555,0.004725
3,23,12.8073,12.8073,15.9690,-0.0078,56.57%,52.93%,0.405205,24.22%,27,9,12.891907,2,6.756782,2.046344,0.597486,0.765385,0.058389,0.656987,8.763033,0.007426
4,22,12.7419,12.7419,15.8464,0.0082,56.36%,44.78%,0.391138,27.61%,43,19,12.891907,2,6.815241,2.003961,0.634872,0.755911,0.059293,0.667165,8.055502,0.006799
5,47,12.8048,12.8048,15.8848,0.0051,56.22%,61.85%,0.411760,31.81%,34,10,12.891907,3,24.571209,2.150971,0.572317,0.721357,0.054521,0.659975,2.793758,0.003758
6,13,12.8453,12.8453,15.8916,0.0033,56.08%,41.42%,0.391498,32.78%,77,40,12.891907,3,6.794567,2.284586,0.687146,0.797569,0.028762,19.797477,4.193945,0.004347
7,31,12.7787,12.7787,15.8321,0.0092,56.06%,54.05%,0.368822,38.78%,77,58,12.891907,2,10.154922,2.539232,0.627916,0.748093,0.037696,0.549346,6.282468,0.004852
8,21,12.8193,12.8193,15.8793,0.0046,55.86%,42.13%,0.354992,20.93%,72,26,12.891907,2,10.664622,2.548827,0.625435,0.717559,0.036357,0.411730,7.086939,0.006361
9,11,12.8601,12.8601,15.9660,-0.0058,55.66%,46.36%,0.335892,34.90%,74,51,12.891907,3,57.111395,2.831450,0.675020,0.789902,0.056453,15.920316,3.644006,0.002162


In [28]:
def fit_and_predict_optuna_day_by_day(train_df, test_df):
    X_train = train_df.drop(columns=EXCLUDE_COLS, errors="ignore")
    y_train = pd.to_numeric(train_df[TARGET_COL], errors="coerce")
    X_test = test_df.drop(columns=EXCLUDE_COLS, errors="ignore")

    model = fit_best_xgb_error_line(
        X_dev=X_train,
        y_dev=y_train,
        sample_weight_dates=train_df["GAME_DATE"],
        sample_weight_lambda=best_trial_lexi.params.get("sample_weight_lambda"),
        trial=best_trial_lexi,
        objective_name="reg:squarederror",
    )
    return model.predict(X_test)

day_by_day_optuna, day_by_day_optuna_thresholds = run_day_by_day_walk_forward_evaluation(
    label="Optuna-selected XGBoost",
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_optuna_day_by_day,
    max_games=TRAIN_GAMES,
)

Optuna-selected XGBoost mean day-by-day OU_Betting_Accuracy: 53.75%


,date,train_n_games,test_n_games,train_start_date,train_end_date,OU_Betting_Accuracy
0,2026-02-24 00:00:00,6750,11,2020-03-08 00:00:00,2026-02-23 00:00:00,81.82%
1,2026-02-25 00:00:00,6750,6,2020-03-10 00:00:00,2026-02-24 00:00:00,33.33%
2,2026-02-26 00:00:00,6750,10,2020-03-10 00:00:00,2026-02-25 00:00:00,33.33%
3,2026-02-27 00:00:00,6750,5,2020-07-31 00:00:00,2026-02-26 00:00:00,40.00%
4,2026-02-28 00:00:00,6750,5,2020-08-01 00:00:00,2026-02-27 00:00:00,40.00%
5,2026-03-01 00:00:00,6750,11,2020-08-02 00:00:00,2026-02-28 00:00:00,63.64%
6,2026-03-02 00:00:00,6750,4,2020-08-04 00:00:00,2026-03-01 00:00:00,75.00%
7,2026-03-03 00:00:00,6750,10,2020-08-05 00:00:00,2026-03-02 00:00:00,90.00%
8,2026-03-04 00:00:00,6750,5,2020-08-06 00:00:00,2026-03-03 00:00:00,60.00%
9,2026-03-05 00:00:00,6750,9,2020-08-07 00:00:00,2026-03-04 00:00:00,88.89%


Optuna-selected XGBoost thresholded walk-forward accuracy


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,1,202,67.6%,55.28%
1,2,119,39.8%,53.39%
2,3,62,20.7%,56.45%


In [29]:
total_df = df_dev.tail(TRAIN_GAMES)

In [30]:
X_dev = total_df.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(total_df[TARGET_COL], errors="coerce")
sample_weight_dates_dev = total_df["GAME_DATE"]


In [31]:
best_model = fit_best_xgb_error_line(
    X_dev=X_dev,
    y_dev=y_dev,
    sample_weight_dates=sample_weight_dates_dev,
    sample_weight_lambda=best_trial_lexi.params.get("sample_weight_lambda"),
    trial=best_trial_lexi,
    objective_name="reg:squarederror",
)

y_pred_test_error = best_model.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_error)
rmse = root_mean_squared_error(y_test_final, y_pred_test_error)
mae = mean_absolute_error(y_test_final, y_pred_test_error)
ou_acc = over_under_betting_accuracy_error_line(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
)
ou_acc_edge_2 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=2,
)
ou_acc_edge_4 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=4,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")
print(f"OU_Betting_Accuracy_Edge_2: {ou_acc_edge_2:.2%}")
print(f"OU_Betting_Accuracy_Edge_4: {ou_acc_edge_4:.2%}")

Final test metrics
MSE: 318.24001
RMSE: 17.83928
MAE: 13.77641
OU_Betting_Accuracy: 49.49%
OU_Betting_Accuracy_Edge_2: 49.17%
OU_Betting_Accuracy_Edge_4: 74.19%


In [32]:
from nba_ou.modeling.modeling import ModelBundleMetadata, ModelInfo, TrainingMetrics

df_to_train_split_rows = df_to_train.copy()
df_to_train_split_rows = df_to_train_split_rows.tail(TRAIN_GAMES)

X_full = df_to_train_split_rows.drop(columns=EXCLUDE_COLS, errors="ignore")
y_full = pd.to_numeric(df_to_train_split_rows[TARGET_COL], errors="coerce")
sample_weight_dates_full = df_to_train_split_rows["GAME_DATE"]

production_model = fit_best_xgb_error_line(
    X_dev=X_full,
    y_dev=y_full,
    sample_weight_dates=sample_weight_dates_full,
    sample_weight_lambda=best_trial_lexi.params.get("sample_weight_lambda"),
    trial=best_trial_lexi,
    objective_name="reg:squarederror",
)

latest_training_date = pd.to_datetime(df_to_train_split_rows["GAME_DATE"]).max()
model_version = latest_training_date.strftime("%d_%m_%y")
model_name = f"all_seasons_xgb_line_error_{model_version}"

metadata = ModelBundleMetadata(
    model_info=ModelInfo(
        name=model_name,
        model_version=model_version,
        model_type="all_seasons_line_error",
        prediction_source="all_seasons_xgb_line_error",
        training_code_tag="1.0",
    ),
    training_metrics=TrainingMetrics(
        best_params=best_trial_lexi.params,
        selected_trial_number=best_trial_lexi.number,
        mean_best_iteration=best_trial_lexi.user_attrs.get("mean_best_iteration"),
        median_best_iteration=best_trial_lexi.user_attrs.get("median_best_iteration"),
        cv_mae=float(best_trial_lexi.user_attrs.get("mean_mae", best_trial_lexi.value)),
        cv_rmse=best_trial_lexi.user_attrs.get("mean_rmse"),
        cv_ou_acc=best_trial_lexi.user_attrs.get("mean_ou_acc"),
        final_test_mae=float(mae),
        final_test_rmse=float(rmse),
        final_test_ou_acc=float(ou_acc),
        nan_threshold=nan_threshold,
        max_na_per_row=max_na_per_row,
        train_date_min=df_to_train_split_rows["GAME_DATE"].min().to_pydatetime(),
        train_date_max=df_to_train_split_rows["GAME_DATE"].max().to_pydatetime(),
        train_games= TRAIN_GAMES,
        sample_weight_lambda_bounds=SAMPLE_WEIGHT_LAMBDA_BOUNDS,
    ),
)

model_path, meta_path = save_model_bundle(
    model=production_model,
    feature_names=list(X_full.columns),
    out_dir="/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/line_error/all_seasons/",
    metadata=metadata,
)

print(
    f"Production model trained on {len(X_full)} rows using fixed n_estimators from median_best_iteration."
)
print("Saved model :", model_path)
print("Saved metadata:", meta_path)

Production model trained on 6750 rows using fixed n_estimators from median_best_iteration.
Saved model : /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/line_error/all_seasons/all_seasons_xgb_line_error_04_04_26.json
Saved metadata: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/line_error/all_seasons/all_seasons_xgb_line_error_04_04_26.meta.json


In [33]:
best_trial_lexi.user_attrs.get("median_best_iteration")


43